In [1]:
# if pandas is not yet installed, open the Python command prompt (in your CPLEX environment) and run
#   pip install pandas

import pandas as pd
#df_latlong = pd.read_csv("latlong.csv").set_index('County Name')
df_demand = pd.read_csv("demand.csv").set_index('County')
df_dist = pd.read_csv("distance_km.csv")

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
Counties = df_demand.index

In [3]:
Demand = {}
for i in df_demand.index:
    Demand[i] = df_demand['Demand'][i]

In [4]:
CanServe = {}
for i in Counties:
    CanServe[i,i] = 1

for i in df_dist.index:
    if df_dist['Distance'][i] <= 150:
        c1 = df_dist['County1'][i]
        c2 = df_dist['County2'][i]
        CanServe[c1,c2] = 1
        if not c1 in Counties:
            print("ERROR: "+c1+" not in Counties")
        if not c2 in Counties:
            print("ERROR: "+c2+" not in Counties")

In [5]:
baselocation = Demand.keys()
destination = Demand.keys()

In [6]:
from docplex.mp.model import Model
mdl = Model()

In [7]:
# variables
Selectbase = mdl.binary_var_dict(Counties, name='select base')
Num_Delivery = mdl.continuous_var_dict(CanServe.keys(), lb=0, name='Number of Deliveries')

In [8]:
# objective
mdl.minimize(mdl.sum(Selectbase[i] for i in Counties))

In [9]:
# Constraints: establish 10 drones in Pennsylvania

#mdl.add_constraint(mdl.sum(Selectbase[i] for i in Counties) == 10)

In [10]:
# Constraints: Each drone base can handle at most 10000 deliveries per month

for i in baselocation:
    add=mdl.add_constraint(mdl.sum(Num_Delivery[i,j] for j in destination if (i,j) in CanServe.keys()) <= 10000*Selectbase[i])


In [11]:
# Demand Constraints at each destination 

for j in destination:
    mdl.add_constraint(mdl.sum(Num_Delivery[i,j] for i in baselocation if (i,j) in CanServe.keys()) == Demand[j])
    
    

In [12]:
# solve
mdl.solve()
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0.0884151,status='integer optimal solution')

In [13]:
mdl.print_solution()

objective: 11
status: OPTIMAL_SOLUTION(2)
  "select base_Adams"=1
  "select base_Allegheny"=1
  "select base_Beaver"=1
  "select base_Carbon"=1
  "select base_Clinton"=1
  "select base_Cumberland"=1
  "select base_Dauphin"=1
  "select base_Erie"=1
  "select base_Lancaster"=1
  "select base_Susquehanna"=1
  "select base_Wayne"=1
  "Number of Deliveries_Adams_Adams"=709.000
  "Number of Deliveries_Allegheny_Allegheny"=5339.000
  "Number of Deliveries_Clinton_Clinton"=323.000
  "Number of Deliveries_Erie_Erie"=2339.000
  "Number of Deliveries_Adams_Berks"=851.000
  "Number of Deliveries_Adams_Lancaster"=4329.000
  "Number of Deliveries_Erie_Mercer"=972.000
  "Number of Deliveries_Erie_Warren"=349.000
  "Number of Deliveries_Erie_Venango"=459.000
  "Number of Deliveries_Erie_Butler"=1533.000
  "Number of Deliveries_Erie_McKean"=363.000
  "Number of Deliveries_Erie_Armstrong"=575.000
  "Number of Deliveries_Erie_Jefferson"=377.000
  "Number of Deliveries_Erie_Beaver"=1422.000
  "Number of D